# Computational Diffraction: Worked Material

**A diffraction image has appeared on the screen. What calculation produced it, and what can we trust about it?**

Chapter 8 developed the propagation operators. Chapter 9 turns them into numerical experiments: specify a physical problem, choose a model, calculate its complex exit field, and follow that field to an observation plane. Each choice leaves a different kind of uncertainty.

We begin with one example throughout: **a normally incident plane wave illuminating a two-dimensional periodic absorbing screen with rectangular clear openings**. We compare a prescribed ideal amplitude mask with the exit field from our scalar Fourier Modal Method (FMM) backend. Then we represent and propagate the fields.

This optional worked example accompanies [Chapter 9](../9_0_From_a_Physical_Model_to_a_Numerical_Experiment.ipynb). It follows the full calculation in code; the individual numerical topics will be developed in the later lessons.

**Prerequisites:** complex fields and coefficients from [0.6](../0_6_From_Complex_Fields_to_Diffraction_Orders.ipynb), the distinction between a mask prescription and material scattering from [0.7](../0_7_Ideal_Binary_Masks_vs_Physical_Gratings.ipynb), and the propagation multipliers in [8.3](../8_3_ASM_Fresnel_and_Fraunhofer_Propagation.ipynb). The FMM matrix derivation is optional; [0.5](../0_5_From_Helmholtz_BVP_to_Diffracted_Field.ipynb) explains its physical role.

In [ ]:
from pathlib import Path
import sys

_support_dir = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within Grating_Physics.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))

import numpy as np
from dataclasses import replace
from IPython.display import display, Image
from io import BytesIO
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from wave_optics.amplitude_grating_2d import (
    absorbing_screen_configuration, AbsorbingComparison2DData,
    field_fourier_coefficients_2d,
)
from wave_optics.scalar_grating_2d import solve_scalar_grating_2d
from wave_optics.absorbing_grating_2d_view import AbsorbingGrating2DExplorer

if 'absorber_comparison' in globals():
    absorber_comparison.close()

## The four-stage calculation

| Stage | Decision | Data passed to the next stage |
| --- | --- | --- |
| **1. Define the mask and illumination** | Geometry, materials, wavelength, incident field, surrounding medium | A physical specification |
| **2. Obtain the exit field** | Prescribe $U_{\mathrm{exit}}=tU_i$, or solve the scalar scattering problem with FMM | Complex exit-field data at a stated plane |
| **3. Represent the exit field** | Spatial samples $U_{\mathrm{exit}}(x_j,y_l)$, or spectral coefficients $C_{mn}$ | A finite numerical representation |
| **4. Propagate the field** | Full ASM, Fresnel, or Fraunhofer within the chosen method's validity range | Complex field, then an intensity prediction |

Representation and approximation are different choices. Spatial and spectral descriptions can encode the same field. Replacing full ASM with Fresnel changes the propagation model, even if both calculations use Fourier coefficients.

**Predict:** if we change only the number of pixels in a field plot, which stage changes? Does the material structure change?

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mn}$) for field coefficients. [Shared convention](../0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


## 1. Specify the physical experiment

The screen repeats with periods $d_x,d_y$. Each unit cell contains an opening of widths $a_x=f_xd_x$ and $a_y=f_yd_y$ in an absorbing background of index $\tilde n=n+i\kappa$. Here $f_x,f_y$ denote **opening fractions**, not spatial frequencies. Both exterior regions and the openings are air.

Use the existing convention $e^{-i\omega t}$. The layer occupies $-h<z<0$, and illumination travels toward $-z$. At normal incidence, $U_i(x,y,z)=e^{-ik_0z}$ with unit amplitude and phase at $z=0$, where $k_0=2\pi/\lambda_0$.

The exit plane is $z=-h$. Define downstream distance **from this exit**, $s\geq0$, so the observation plane is $z=-h-s$. All lengths in this example are in **micrometres**.

We solve scalar Helmholtz with continuity of $U$ and its normal derivative at material interfaces. Positive $\kappa$ represents passive absorption. These are the scalar assumptions established earlier in the course; the material indices are illustrative.

**What will we calculate?** A field map over one unit cell at the observation plane, its relative intensity $|U|^2$, and the physical screen's reflected, transmitted, and absorbed scalar power fractions. The structure and illumination extend periodically; this is not a finite array of holes.

In [ ]:
# Physical inputs: changing these changes the optical problem.
physical_inputs = dict(
    wavelength=0.633, period_x=1.5, period_y=1.5, thickness=0.3,
    fill_x=0.5, fill_y=0.5,
    n_incident=1.0, n_output=1.0, n_inclusion=1.0,
    n_background=1.5 + 0.8j,
    kx_incident=0.0, ky_incident=0.0,
)
s_observe = 0.633  # Distance below the exit, in um.

# Numerical controls: changing these should refine the same physical problem.
cutoffs = dict(Mx=3, My=3)
Nx, Ny = 64, 64  # Samples used to represent one unit cell.
config = absorbing_screen_configuration(**physical_inputs, **cutoffs)
z_observe = -config.thickness - s_observe

print(f'Clear opening: {config.fill_x * config.period_x:.3f} x '
      f'{config.fill_y * config.period_y:.3f} um')
print(f'Exit z = {-config.thickness:.3f} um; observation z = {z_observe:.3f} um')
print(f'Retained Fourier harmonics: {(2 * config.Mx + 1) * (2 * config.My + 1)}')

Keep physical inputs and numerical controls separate. Increasing $M_x,M_y$ adds harmonics to the material-scattering calculation. Increasing $N_x,N_y$ gives more spatial samples of the resulting field. They do different jobs.

The starting cutoffs are deliberately modest so this introduction runs quickly. They define an initial approximation, not a certified accuracy. At the end we will refine them without changing the structure.

For this introductory experiment, keep normal incidence and equal air indices in the openings and exterior. Later notebooks develop oblique incidence and Gaussian illumination.

## 2. Obtain the exit field in two ways

**Ideal thin amplitude mask.** Let $\chi_{\mathrm{open}}(x,y)$ equal one in each rectangular opening and zero elsewhere. Place the prescription at the same exit plane as the physical screen:

$$U_{\mathrm{exit}}^{\mathrm{ideal}}(x,y)
=\chi_{\mathrm{open}}(x,y)\,e^{ik_0h}.$$

The factor $e^{ik_0h}$ continues the incident wave through air from $z=0$ to $z=-h$. It keeps the absolute phase reference consistent. It is not a fitted correction. This prescription supplies a boundary field; it does not solve for reflection or physical absorption.

**Physical scalar screen.** FMM takes the material geometry and incident wave, solves the coupled Fourier harmonics inside the layer, and matches both interfaces. Its transmitted coefficients are outputs of that scattering problem. Field may leak through the absorber, and scattering can change the field in and around the openings.

Both fields will use the **same exit plane, incident phase reference, and downstream propagation operator**. Their difference tests the consequences of the mask prescription; it is not, by itself, evidence of numerical error.

In [ ]:
# Solve the material problem once and construct the matching ideal prescription.
comparison_data = AbsorbingComparison2DData(config)
solution = comparison_data.solution
ideal = comparison_data.ideal
C_exit = solution.transmission
C_exit_ideal = ideal.transmission

# Divide by the same continued-incident reference to inspect transmission phases.
incident_exit_reference = ideal.incident_reference
j00 = solution.incident_index
print(f'FMM C_00 / reference:   {C_exit[j00] / incident_exit_reference:+.6f}')
print(f'Ideal C_00 / reference: {C_exit_ideal[j00] / incident_exit_reference:+.6f}')
print('Both coefficient arrays have shape:', C_exit.shape)

For the ideal mask, the normalized zero-order coefficient is the open area fraction $f_xf_y=0.25$. The FMM result need not equal it: the same intended amplitude pattern has been represented by a finite-thickness material structure.

An FMM exit field calculated for this plane wave is specific to this illumination. Multiplying it by a Gaussian envelope would introduce another approximation. A later notebook will obtain Gaussian illumination by coherently combining the screen's responses to its incident angular spectrum.

## 3. Represent the same exit field in space and in orders

At normal incidence, the retained field is

$$U_{\mathrm{exit}}^{(M)}(x,y)=
\sum_{m=-M_x}^{M_x}\sum_{n=-M_y}^{M_y}
C_{mn}\,e^{i(k_{x,m}x+k_{y,n}y)},\qquad
k_{x,m}=\frac{2\pi m}{d_x},\quad k_{y,n}=\frac{2\pi n}{d_y}.$$

The backend returns `orders[j] = (m, n)` and `transmission[j] = C_mn`. The order arrays are flattened with $m$ outermost and $n$ innermost. Spatial arrays use the shape **`(Ny, Nx)`**; the first index runs along $y$.

Evaluate this finite sum on an endpoint-excluded grid covering one unit cell. The pixel centres lie halfway between the cell boundaries. A complex FFT2 should recover the same retained coefficients, including their phases. The helper corrects for the coordinate origin.

**Predict:** will transforming $|U|$ or $|U|^2$ recover $C_{mn}$? What information would have been discarded?

In [ ]:
x = ((np.arange(Nx) + 0.5) / Nx - 0.5) * config.period_x
y = ((np.arange(Ny) + 0.5) / Ny - 0.5) * config.period_y
U_exit = solution.reconstruct(x, y, [-config.thickness])[0]
C_from_grid = field_fourier_coefficients_2d(U_exit, x, y, config)
fft_error = np.max(np.abs(C_from_grid - C_exit))

print(f'Spatial array: {U_exit.shape}; order pairs: {solution.orders.shape}')
print(f'dx = {config.period_x / Nx:.5f} um; dy = {config.period_y / Ny:.5f} um')
print(f'Maximum |C FFT2 - C direct| = {fft_error:.2e}')
np.testing.assert_allclose(C_from_grid, C_exit, rtol=0, atol=1e-12)

The FFT agreement checks a change of representation of a **finite harmonic solution**. It does not independently verify the material model, FMM truncation, or its agreement with experiment.

Keep $N_x>2M_x$ and $N_y>2M_y$ for unambiguous recovery of all retained orders. These bounds concern the finite sum at normal incidence. They are not a universal sampling prescription for a sharp ideal mask, whose exact Fourier series contains infinitely many harmonics.

Increasing the number of plotted pixels at fixed $M_x,M_y$ samples the same truncated field more densely. It cannot add harmonics that the scattering solve omitted. Notebook 9.2 will develop the general connection between spatial and spectral grids.

## 4. Propagate fixed exit data

In homogeneous output air, each retained order evolves independently:

$$C_{mn}(s)=C_{mn}(0)\,e^{i\beta_{mn}s},\qquad
\beta_{mn}=\sqrt{k_0^2-k_{x,m}^2-k_{y,n}^2}.$$

Choose the outgoing square-root branch: $\beta_{mn}$ is nonnegative real for propagating orders and positive imaginary for evanescent orders. Thus propagation changes the phase of propagating components and attenuates evanescent ones. Recall that $s$ is measured from the exit; the backend's signed transmitted wavevector is $k_z=-\beta_{mn}$.

This is **full angular-spectrum propagation of discrete periodic orders**. Because their coefficients are already available, no FFT is needed to propagate them. Here the FFT2 above is only a representation check. The two exit models receive the same factors.

Fresnel replaces the longitudinal dispersion by its paraxial approximation. Fraunhofer makes an additional far-field approximation. We will compare these later. For an infinite periodic field, far-field orders are discrete directions; finite-width spots require finite illumination or a finite aperture.

In [ ]:
C_observe, C_observe_ideal = comparison_data.coefficients(s_observe)
propagation_factors = np.exp(1j * solution.beta_output * s_observe)
np.testing.assert_allclose(C_observe, C_exit * propagation_factors, rtol=0, atol=1e-12)

print('Order     Type           |propagation factor|')
for pair in ((0, 0), (1, 0), (2, 2)):
    found = np.flatnonzero(np.all(solution.orders == pair, axis=1))
    if found.size:
        j = int(found[0])
        print(f'{pair!s:9s} {solution.output_kinds[j]:13s} {abs(propagation_factors[j]):.6f}')

U_observe = solution.reconstruct(x, y, [z_observe])[0]
U_observe_ideal = ideal.reconstruct(x, y, [z_observe])[0]
I_observe = np.abs(U_observe)**2
I_observe_ideal = np.abs(U_observe_ideal)**2

### What would an ideal intensity measurement show?

The following maps show relative scalar intensity $|U|^2/|U_i|^2$, with unit incident amplitude. Both predictions use the same illumination and a common colour scale. No curve or image is normalized to its own maximum.

This is a local intensity proxy at the observation plane. A full camera model would also specify collection optics, pixel integration and detector response. The transmitted power fraction $T$ is a different quantity: it includes the normal flux weights of the orders.

The displayed cell repeats throughout the plane. Its edges are not the edges of a finite physical screen.

In [ ]:
observation_figure = Figure(figsize=(11.4, 3.8), dpi=110, constrained_layout=True)
FigureCanvasAgg(observation_figure)
observation_axes = observation_figure.subplots(1, 3)
extent = [-config.period_x / 2, config.period_x / 2,
          -config.period_y / 2, config.period_y / 2]
common_max = max(0.01, float(I_observe.max()), float(I_observe_ideal.max()))

for ax, values, title in zip(observation_axes[:2],
        (I_observe, I_observe_ideal), ('Physical scalar screen', 'Ideal amplitude mask')):
    image = ax.imshow(values, origin='lower', extent=extent, cmap='magma',
                      vmin=0, vmax=common_max, interpolation='nearest')
    ax.set(title=title, xlabel=r'$x$ ($\mu$m)', ylabel=r'$y$ ($\mu$m)')
observation_figure.colorbar(image, ax=list(observation_axes[:2]), shrink=0.8,
                          label=r'$|U|^2 / |U_i|^2$')
difference = observation_axes[2].imshow(np.abs(I_observe - I_observe_ideal),
    origin='lower', extent=extent, cmap='viridis', vmin=0, interpolation='nearest')
observation_axes[2].set(title='Absolute intensity difference',
    xlabel=r'$x$ ($\mu$m)', ylabel=r'$y$ ($\mu$m)')
observation_figure.colorbar(difference, ax=observation_axes[2], shrink=0.8,
                          label=r'$|I_{\rm FMM}-I_{\rm ideal}|$')
observation_figure.suptitle(f'One periodic cell; distance below exit s = {s_observe:.3f} um')
observation_png = BytesIO()
observation_figure.savefig(observation_png, format='png', facecolor='white')
display(Image(data=observation_png.getvalue()))

## 5. Follow the complete calculation interactively

The explorer reuses the FMM solution already obtained. Changing the selected order or observation distance updates the field; it does not solve a new material problem. Its default distance is zero, at the exit plane; the static maps above used `s_observe`.

1. **Structure and exit field:** compare the clear opening, the absorbing material, and the calculated field. The ideal binary prescription and its retained Fourier reconstruction are shown separately.
2. **Complex order lattice:** select $(0,0)$, $(1,0)$, then $(2,2)$. Inspect the amplitude, phase, and propagating/evanescent classification. Move the observation plane below the exit.
3. **Propagation:** compare both field maps at the same distance. Their amplitudes share one scale. The differences began with their exit fields.

The coefficient phase plots use the same continued-incident exit reference. Phases are hidden when the relevant amplitude is below $10^{-6}$ of the unit incident field. Rerunning this cell replaces the previous explorer cleanly.

In [ ]:
if 'absorber_comparison' in globals():
    absorber_comparison.close()
absorber_comparison = AbsorbingGrating2DExplorer(comparison_data)
display(absorber_comparison.ui)

## 6. Ask what each numerical check establishes

| Question | Useful check | What remains to be checked |
| --- | --- | --- |
| Did spatial and spectral data encode the same retained field? | Recover $C_{mn}$ from the complex-field FFT2 | Material-model accuracy and order convergence |
| Is the passive scalar power budget consistent? | Compute $R$, $T$, and independently integrated $A$; check $R+T+A\approx1$ | Accuracy of individual coefficients and local fields |
| Is FMM truncation adequate for our observable? | Increase $M_x,M_y$ while keeping physical inputs fixed | Further refinements and the model's physical assumptions |
| Is the spatial map adequately sampled? | Refine $N_x,N_y$ at fixed harmonic coefficients | Harmonics missing from the FMM solve |
| Is a propagation approximation adequate? | Compare it with full propagation for the same exit field | The accuracy of that supplied exit field |

For example, compare the mean sampled intensity with the sum of squared order magnitudes. This is Parseval's identity for the retained periodic field. It is not a power-transmission formula: normal flux contains additional longitudinal-wavevector weights.

In [ ]:
print(f'R = {solution.reflectance:.8f}')
print(f'T = {solution.transmittance:.8f}')
print(f'A = {solution.absorption:.8f}  (volume integral, not 1-R-T)')
print(f'|R + T + A - 1| = {abs(solution.energy_balance - 1):.2e}')

mean_intensity_grid = float(np.mean(I_observe))
mean_intensity_orders = float(np.sum(np.abs(C_observe)**2))
print(f'Mean |U|^2: grid = {mean_intensity_grid:.8f}; orders = {mean_intensity_orders:.8f}')
np.testing.assert_allclose(mean_intensity_grid, mean_intensity_orders, rtol=0, atol=1e-12)

### One controlled refinement

Keep the physical specification and observation plane fixed. Increase both FMM cutoffs, and inspect **one complex coefficient** and **the intensity at the cell centre**, as well as total transmitted power. Sampling more points in a plot cannot perform this refinement.

The first row reuses the existing solution; only the next two rows require additional solves. This short sequence illustrates how to investigate convergence. It is not an error bound or a universal choice of cutoff. Later we will refine the two transverse cutoffs independently and examine more demanding near-field observables.

In [ ]:
refinement_solutions = [solution]
for increment in (1, 2):
    trial_config = replace(config, Mx=config.Mx + increment, My=config.My + increment)
    refinement_solutions.append(solve_scalar_grating_2d(trial_config))

print('Mx  My     K         C_00 at exit               T           I(0,0,s)      |R+T+A-1|')
for trial in refinement_solutions:
    centre = trial.reconstruct([0.0], [0.0], [z_observe])[0, 0, 0]
    c00 = trial.transmission[trial.incident_index]
    print(f'{trial.configuration.Mx:2d}  {trial.configuration.My:2d}  {len(trial.orders):4d}   '
          f'{c00.real:+.6f}{c00.imag:+.6f}j   {trial.transmittance:.6f}   '
          f'{abs(centre)**2:.6f}       {abs(trial.energy_balance - 1):.1e}')

**Read the table:** a nearly perfect power budget can coexist with changes in $C_{00}$ or local intensity. Each truncated model can conserve scalar power while approximating the material-scattering problem differently. Use an observable-specific tolerance and further refinements before reporting quantitative accuracy.

The explorer and static maps deliberately remain attached to the original solution. Running this comparison does not silently replace the calculation we have been inspecting.

Three uncertainty sources should now be distinguishable: **model assumptions** (for example, ideal mask versus physical scalar screen), **numerical representation** (harmonic and spatial sampling), and **propagation approximations** (full ASM versus Fresnel/Fraunhofer). A complete calculation records all three.

## Learning checkpoint

1. Does changing the index of the absorber refine the numerical solution, or change the physical problem?
2. Why does a tiny FFT2 coefficient error not establish that the FMM cutoff is sufficient?
3. Why can the ideal-mask and physical-screen results disagree even when both numerical calculations are accurate?
4. Does propagating the known periodic coefficients require an FFT?
5. Why is an intensity image over one cell insufficient to certify a finite-beam calculation?

<details><summary>Check your reasoning</summary>

Changing the material index changes the physical problem. The FFT2 check only compares representations of the same retained sum. The ideal prescription and the material scattering solve are different models of the intended mask. Known periodic coefficients propagate directly through their ASM factors. A finite beam introduces a finite illuminated extent and a broadened spectrum, whose sampling and window requirements are absent from an infinite-periodic-cell calculation.

</details>

## Where the computational series goes next

| Planned lesson | New numerical question |
| --- | --- |
| **[9.1 - Defining the Complex Exit Field](../9_1_Defining_the_Complex_Exit_Field.ipynb)** | What is the exit field, which representation are we using, and what repeats? |
| **9.2 - Sampling Complex Fields and Their Spectra** | Connect physical coordinates, array spacing, frequency bins, and bandwidth. |
| **9.3 - Computing the Exit Field: Thin Masks and Scalar FMM** | Compare a prescribed boundary field with material scattering and study order convergence. |
| **9.4 - Plane-Wave Illumination of Periodic Gratings** | Reconstruct and propagate discrete orders; separate physical periodicity from numerical artifacts. |
| **9.5 - Gaussian Illumination and Finite Computational Windows** | Resolve finite illumination and the resulting broadened spectral peaks. |
| **9.6 - Reliable FFT Propagation** | Examine transfer-function sampling, padding, wraparound, and band-limiting. |
| **9.7 - Gaussian Illumination with the FMM Backend** | Coherently integrate the material response over the incident angular spectrum. |
| **9.8 - Choosing the Observation Grid** | Introduce shifted, scaled, and nonuniform propagation when the application requires them. |

The finite-beam lessons will treat finite windows as a numerical choice. Here the periodic repetition is physical. Padding a single periodic cell with zeros would change its prescribed field.

## References and navigation

The notebook uses the [2D scalar FMM backend](../wave_optics/scalar_grating_2d.md), including its stated single-layer scalar assumptions, passive material convention, and initial regression checks. Independent external solver benchmarking remains deferred.

For the numerical questions developed in the coming lessons:

1. **F. Shen and A. Wang (2006), "Fast-Fourier-transform based numerical integration method for the Rayleigh-Sommerfeld diffraction formula," Applied Optics 45, 1102-1110.** [DOI](https://doi.org/10.1364/AO.45.001102). Read Section 4 for sampling, computation windows, and differences between FFT-AS and FFT-DI.
2. **K. Matsushima and T. Shimobaba (2009), "Band-Limited Angular Spectrum Method for Numerical Simulation of Free-Space Propagation in Far and Near Fields," Optics Express 17, 19662-19673.** [DOI](https://doi.org/10.1364/OE.17.019662). Read Sections 2.2 and 3 for circular convolution and transfer-function sampling. These finite-window issues are developed later in this series.
3. **W. Zhang, H. Zhang, and G. Jin (2020), "Frequency sampling strategy for numerical diffraction calculations," Optics Express 28, 39916-39932.** [DOI](https://doi.org/10.1364/OE.413636). Read the introduction and conclusion for the relation between physical bandwidth, sampling, and algorithm choice.

The next lesson develops sampling before adding further propagation algorithms.

[Previous: 8.3 - ASM, Fresnel and Fraunhofer](../8_3_ASM_Fresnel_and_Fraunhofer_Propagation.ipynb) | [Propagation chapter](../8_0_Introduction_to_Field_Propagation.ipynb)